# Member 5 — Backend Aggregation Analysis

This notebook uses `listenbrainz_final_cleaned.csv`, prepared by Member 3, to create reusable and chart-ready datasets for:

- audience growth
- listener engagement
- listening activity over time
- artist performance

The original cleaned dataset is not modified.

In [1]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("../listenbrainz_final_cleaned.csv")

listens = pd.read_csv(DATA_PATH)

print("Dataset shape:", listens.shape)
print("\nColumn names:")
print(listens.columns.tolist())

print("\nData types:")
print(listens.dtypes)

listens.head()

Dataset shape: (1940, 11)

Column names:
['username', 'listened_at', 'track_name', 'artist_name', 'release_name', 'recording_msid', 'artist_mbids', 'recording_mbid', 'release_mbid', 'listened_at_utc', 'artist_mbids_parsed']

Data types:
username               str
listened_at            str
track_name             str
artist_name            str
release_name           str
recording_msid         str
artist_mbids           str
recording_mbid         str
release_mbid           str
listened_at_utc        str
artist_mbids_parsed    str
dtype: object


,username,listened_at,track_name,artist_name,release_name,recording_msid,artist_mbids,recording_mbid,release_mbid,listened_at_utc,artist_mbids_parsed
0,iliekcomputers,2024-12-16 08:04:37,Take Care,"Drake, Rihanna",Take Care (Deluxe),a9dccce2-eb7b-4e87-ba43-0cf9219997b7,"['9fff2f8a-21e6-47de-a2b8-7f449929d43f', '73e5...",a23ccdb8-765f-4b7d-b47a-63191b2f856f,02f7e348-b1ae-4c4e-a057-b97f1f56e53c,2024-12-16T08:04:37+00:00,"[""9fff2f8a-21e6-47de-a2b8-7f449929d43f"",""73e5e..."
1,iliekcomputers,2024-12-16 05:56:01,Us Against The World,Jungle,Volcano,c97b7e4d-17f9-4a1b-a26e-ec107bae5db0,['6bbb3983-ce8a-4971-96e0-7cae73268fc4'],14f9f386-3df5-4be3-b92c-e473b0afd98e,a041661f-d1c4-4398-b23d-6bd156bf1aed,2024-12-16T05:56:01+00:00,"[""6bbb3983-ce8a-4971-96e0-7cae73268fc4""]"
2,iliekcomputers,2024-12-16 05:44:12,Love on the Ground,Cannons,Shadows,000c52a6-b412-4a92-a948-ce6fc35bd2e6,['eb9eb9f8-c691-4fef-9bdf-0cbffbfa42ee'],2b49aa61-3d97-4555-811c-204f56d0a0fa,f9ed6b4a-3357-4f0a-907a-5f73cea0b72e,2024-12-16T05:44:12+00:00,"[""eb9eb9f8-c691-4fef-9bdf-0cbffbfa42ee""]"
3,iliekcomputers,2024-12-16 05:40:30,Shadows,Cannons,Shadows,6ddd1875-d288-40f8-ace6-8e3b598eaa59,['eb9eb9f8-c691-4fef-9bdf-0cbffbfa42ee'],16279f1d-ddc5-455c-b3db-b70a094af533,f9ed6b4a-3357-4f0a-907a-5f73cea0b72e,2024-12-16T05:40:30+00:00,"[""eb9eb9f8-c691-4fef-9bdf-0cbffbfa42ee""]"
4,iliekcomputers,2024-12-16 05:36:34,Bright Lights,Cannons,Shadows,769d59c9-be4f-4097-8607-ba8deb018321,['eb9eb9f8-c691-4fef-9bdf-0cbffbfa42ee'],b0f87b27-ccfe-423a-8428-8af152211fc4,f9ed6b4a-3357-4f0a-907a-5f73cea0b72e,2024-12-16T05:36:34+00:00,"[""eb9eb9f8-c691-4fef-9bdf-0cbffbfa42ee""]"


## 1. Prepare and inspect the cleaned data

Convert the UTC listening timestamp into a datetime value and examine the dataset's user and time coverage before building aggregations.

In [2]:
# Work on a copy so the original cleaned dataset remains unchanged
analysis_df = listens.copy()

# Convert the analysis-ready UTC timestamp to datetime
analysis_df["listened_at_utc"] = pd.to_datetime(
    analysis_df["listened_at_utc"],
    utc=True,
    errors="coerce"
)

# Create chart-friendly time fields
analysis_df["date"] = analysis_df["listened_at_utc"].dt.date
analysis_df["week"] = (
    analysis_df["listened_at_utc"]
    .dt.to_period("W")
    .dt.start_time
)
analysis_df["month"] = (
    analysis_df["listened_at_utc"]
    .dt.to_period("M")
    .dt.start_time
)

print("Total listening events:", len(analysis_df))
print("Unique users:", analysis_df["username"].nunique())
print("Unique artists:", analysis_df["artist_name"].nunique())
print("Unique tracks:", analysis_df["track_name"].nunique())
print("Earliest timestamp:", analysis_df["listened_at_utc"].min())
print("Latest timestamp:", analysis_df["listened_at_utc"].max())
print("Invalid timestamps:", analysis_df["listened_at_utc"].isna().sum())

Total listening events: 1940
Unique users: 5
Unique artists: 756
Unique tracks: 1308
Earliest timestamp: 2019-03-07 13:17:39+00:00
Latest timestamp: 2026-09-21 03:51:02+00:00
Invalid timestamps: 0


/var/folders/9k/3n_qq8ss4vz7qkzj89c9_f6h0000gn/T/ipykernel_71592/881067248.py:15: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  .dt.to_period("W")
/var/folders/9k/3n_qq8ss4vz7qkzj89c9_f6h0000gn/T/ipykernel_71592/881067248.py:20: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  .dt.to_period("M")


In [3]:
user_coverage = (
    analysis_df.groupby("username", as_index=False)
    .agg(
        first_listen=("listened_at_utc", "min"),
        last_listen=("listened_at_utc", "max"),
        total_listens=("username", "size"),
        active_days=("date", "nunique"),
        unique_artists=("artist_name", "nunique"),
        unique_tracks=("track_name", "nunique")
    )
    .sort_values("total_listens", ascending=False)
)

user_coverage

,username,first_listen,last_listen,total_listens,active_days,unique_artists,unique_tracks
0,H-music,2019-07-15 16:56:10+00:00,2019-11-15 18:03:47+00:00,500,28,122,190
2,iliekcomputers,2024-11-16 01:49:07+00:00,2024-12-16 08:04:37+00:00,500,24,240,320
4,s_john_s,2019-03-07 13:17:39+00:00,2019-03-20 12:58:48+00:00,500,8,289,425
1,John V,2026-09-17 01:31:34+00:00,2026-09-21 03:51:02+00:00,226,5,33,184
3,music_af,2025-09-17 13:25:40+00:00,2026-03-25 07:32:42+00:00,214,33,77,192


## 2. Audience growth aggregation

Because this dataset contains only five sampled users with different observation periods, the following measures describe observed sample activity rather than total ListenBrainz platform growth.

Metrics:

- `active_listeners`: unique sampled users active during the month
- `new_listeners`: users appearing in the sample for the first time
- `cumulative_listeners`: cumulative number of users observed
- `listener_change`: change in active listeners from the previous month
- `listener_growth_pct`: percentage change from the previous month

In [4]:
from aggregation_functions import build_audience_growth


In [5]:
audience_growth = build_audience_growth(listens)

print(audience_growth.shape)
audience_growth[
    (audience_growth["active_listeners"] > 0)
    | (audience_growth["new_listeners"] > 0)
]

(91, 6)


,month,active_listeners,new_listeners,cumulative_listeners,listener_change,listener_growth_pct
0,2019-03-01,1,1,1,NaN,NaN
4,2019-07-01,1,1,2,1.0,<NA>
5,2019-08-01,1,0,2,0.0,0.0
6,2019-09-01,1,0,2,0.0,0.0
7,2019-10-01,1,0,2,0.0,0.0
8,2019-11-01,1,0,2,0.0,0.0
68,2024-11-01,1,1,3,1.0,<NA>
69,2024-12-01,1,0,3,0.0,0.0
78,2025-09-01,1,1,4,1.0,<NA>
79,2025-10-01,1,0,4,0.0,0.0


In [6]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

audience_growth.to_csv(
    OUTPUT_DIR / "audience_growth.csv",
    index=False
)

print("Saved:", OUTPUT_DIR / "audience_growth.csv")

Saved: outputs/audience_growth.csv


## 3. Listener engagement aggregation

Create one row per listener with listening volume, active days, content diversity, and repeat-listening measures.

In [7]:
from aggregation_functions import build_listener_engagement


In [8]:
listener_engagement = build_listener_engagement(listens)

listener_engagement

,username,first_listen,last_listen,total_listens,active_days,unique_artists,unique_tracks,unique_releases,observation_days,listens_per_active_day,repeat_listens,repeat_listen_pct
0,H-music,2019-07-15 16:56:10+00:00,2019-11-15 18:03:47+00:00,500,28,122,190,52,124,17.86,310,62.00
1,iliekcomputers,2024-11-16 01:49:07+00:00,2024-12-16 08:04:37+00:00,500,24,240,320,231,31,20.83,180,36.00
2,s_john_s,2019-03-07 13:17:39+00:00,2019-03-20 12:58:48+00:00,500,8,289,425,194,14,62.50,75,15.00
3,John V,2026-09-17 01:31:34+00:00,2026-09-21 03:51:02+00:00,226,5,33,184,0,5,45.20,42,18.58
4,music_af,2025-09-17 13:25:40+00:00,2026-03-25 07:32:42+00:00,214,33,77,192,98,190,6.48,22,10.28


In [9]:
listener_engagement.to_csv(
    OUTPUT_DIR / "listener_engagement.csv",
    index=False
)

print("Saved:", OUTPUT_DIR / "listener_engagement.csv")

Saved: outputs/listener_engagement.csv


## 4. Daily listener activity

Create one row for each user-date combination. This table can be used directly for dashboard line charts, user filters, and activity comparisons.

In [10]:
from aggregation_functions import build_daily_activity


In [11]:
daily_activity = build_daily_activity(listens)

print("Shape:", daily_activity.shape)
daily_activity.head(10)

Shape: (98, 6)


,date,username,total_listens,unique_artists,unique_tracks,unique_releases
0,2019-03-07,s_john_s,11,10,10,8
1,2019-03-08,s_john_s,43,41,41,25
2,2019-03-09,s_john_s,134,124,132,73
3,2019-03-10,s_john_s,112,35,109,25
4,2019-03-12,s_john_s,34,25,33,20
5,2019-03-16,s_john_s,133,108,124,70
6,2019-03-17,s_john_s,11,11,11,6
7,2019-03-20,s_john_s,22,20,20,14
8,2019-07-15,H-music,63,1,1,0
9,2019-07-17,H-music,2,2,2,0


In [12]:
daily_activity.to_csv(
    OUTPUT_DIR / "daily_listener_activity.csv",
    index=False
)

print("Saved:", OUTPUT_DIR / "daily_listener_activity.csv")

Saved: outputs/daily_listener_activity.csv


## 5. Artist performance aggregation

Create one row per artist with total listening volume, audience reach, content diversity, and activity period. Artist collaborations remain grouped according to the cleaned `artist_name` value.

In [13]:
from aggregation_functions import build_artist_performance


In [14]:
artist_performance = build_artist_performance(listens)

print("Shape:", artist_performance.shape)
artist_performance.head(10)

Shape: (756, 10)


,artist_name,total_listens,unique_listeners,unique_tracks,unique_releases,active_days,first_listen,last_listen,listens_per_listener,listen_share_pct
0,Arthrn,176,1,40,8,18,2019-07-23 12:37:53+00:00,2019-11-05 23:11:01+00:00,176.0,9.07
1,Roseboy666,116,1,1,0,4,2019-07-15 16:56:10+00:00,2019-09-20 14:32:48+00:00,116.0,5.98
2,Alan Silvestri - Topic,77,1,77,0,1,2026-09-18 12:12:41+00:00,2026-09-18 15:59:44+00:00,77.0,3.97
3,Kanye West,34,1,18,6,3,2024-11-16 01:49:07+00:00,2024-12-10 19:59:48+00:00,34.0,1.75
4,Takanashi Yasuharu / -yaiba-,33,1,32,4,3,2019-03-10 09:12:50+00:00,2019-03-16 09:49:36+00:00,33.0,1.70
5,Yasuharu Takanashi / -yaiba-,28,1,27,3,3,2019-03-10 09:16:57+00:00,2019-03-16 09:47:05+00:00,28.0,1.44
6,ministerio3d,25,1,1,0,1,2026-09-19 17:58:11+00:00,2026-09-19 21:45:46+00:00,25.0,1.29
7,Toshiro Masuda,24,1,23,3,3,2019-03-10 09:17:38+00:00,2019-03-16 09:42:07+00:00,24.0,1.24
8,Diante do Trono,16,1,12,0,2,2026-09-20 03:42:26+00:00,2026-09-21 03:51:02+00:00,16.0,0.82
9,Diljit Dosanjh,16,1,9,4,5,2024-11-20 09:11:02+00:00,2024-12-13 06:25:44+00:00,16.0,0.82


In [15]:
artist_performance.to_csv(
    OUTPUT_DIR / "artist_performance.csv",
    index=False
)

print("Saved:", OUTPUT_DIR / "artist_performance.csv")

Saved: outputs/artist_performance.csv


In [16]:
sorted(path.name for path in OUTPUT_DIR.glob("*.csv"))

['artist_engagement.csv',
 'artist_monthly_growth.csv',
 'artist_performance.csv',
 'audience_growth.csv',
 'daily_listener_activity.csv',
 'listener_engagement.csv']

In [17]:
from aggregation_functions import parse_artist_mbids


artist_id_check = listens["artist_mbids_parsed"].apply(parse_artist_mbids)

print("Total rows:", len(listens))
print("Rows with artist MBIDs:", artist_id_check.str.len().gt(0).sum())
print("Rows without artist MBIDs:", artist_id_check.str.len().eq(0).sum())
print("Maximum artists on one listen:", artist_id_check.str.len().max())
print("Unique resolved artist MBIDs:", len(set(artist_id_check.explode().dropna())))

Total rows: 1940
Rows with artist MBIDs: 1181
Rows without artist MBIDs: 759
Maximum artists on one listen: 6
Unique resolved artist MBIDs: 606


## 6. Build the artist-listen event table

Explode `artist_mbids_parsed` so that each resolved artist MBID receives one row for each associated listening event. Unresolved artist records are excluded from artist-level rankings, following the dashboard specification.

In [18]:
from aggregation_functions import build_artist_listen_events


In [19]:
artist_events = build_artist_listen_events(listens)

print("Exploded artist-event rows:", len(artist_events))
print(
    "Original listens represented:",
    artist_events["listen_event_id"].nunique()
)
print(
    "Unique artist MBIDs:",
    artist_events["artist_mbid"].nunique()
)

artist_events.head()

Exploded artist-event rows: 1479
Original listens represented: 1181
Unique artist MBIDs: 606


,artist_mbids_parsed,track_name,recording_mbid,listened_at_utc,artist_name,username,listen_event_id,artist_mbid,date,month
0,"[""ef30fb35-acf6-4bdc-aeb3-fb9b7d6b813d""]",Reality,bf2181ec-ebb3-4d48-af2b-3dc096bcc014,2019-03-07 13:17:39+00:00,Richard Sanderson,s_john_s,1713,ef30fb35-acf6-4bdc-aeb3-fb9b7d6b813d,2019-03-07,2019-03-01
1,"[""e57f0cac-4f56-473c-8d7e-d93f753fd586""]",Abracadabra,c0dfa075-6172-46ba-8059-4a92443a7bf0,2019-03-07 13:21:51+00:00,Steve Miller Band,s_john_s,1712,e57f0cac-4f56-473c-8d7e-d93f753fd586,2019-03-07,2019-03-01
2,"[""577d92bd-32ac-48b6-86df-4db6b0b52fbf""]",Oh! Carol,b69ea195-7be6-48c6-92ab-79d051f2937e,2019-03-07 13:25:35+00:00,Neil Sedaka,s_john_s,1711,577d92bd-32ac-48b6-86df-4db6b0b52fbf,2019-03-07,2019-03-01
3,"[""6286e04b-7bc1-4f22-bca5-fdd34d1b0d93""]",The Riddle,32ed6b51-b3ae-4910-a769-79ea59a72417,2019-03-07 13:27:52+00:00,Nik Kershaw,s_john_s,1710,6286e04b-7bc1-4f22-bca5-fdd34d1b0d93,2019-03-07,2019-03-01
4,"[""7d46fd76-5b71-4d7e-8cef-1ad8b135a109""]",Julie Ann,467bc324-6d7f-4243-8f64-0a7f1869df5c,2019-03-07 13:31:32+00:00,Kenny,s_john_s,1709,7d46fd76-5b71-4d7e-8cef-1ad8b135a109,2019-03-07,2019-03-01


## 7. Artist growth over time

Aggregate monthly unique listeners by resolved artist MBID. Listener growth is calculated only when an artist has observations in two consecutive months.

In [20]:
from aggregation_functions import build_artist_monthly_growth


In [21]:
artist_monthly_growth = build_artist_monthly_growth(
    artist_events
)

print("Shape:", artist_monthly_growth.shape)
print(
    "Artists with consecutive-month comparisons:",
    artist_monthly_growth.loc[
        artist_monthly_growth["has_consecutive_month"],
        "artist_mbid"
    ].nunique()
)

artist_monthly_growth.sort_values(
    ["unique_listeners", "total_listens"],
    ascending=False
).head(10)

Shape: (691, 11)
Artists with consecutive-month comparisons: 59


,artist_mbid,month,artist_credit,unique_listeners,total_listens,unique_tracks,previous_month,previous_unique_listeners,has_consecutive_month,listener_change,listener_growth_pct
104,25d54fa4-0afa-4e47-a1ce-e2cfe5948fda,2019-08-01,Arthrn,1,57,9,2019-07-01,1.0,True,0.0,0.0
61,164f0d73-1234-4e2c-8743-d77bf2191051,2024-11-01,Kanye West,1,53,34,NaT,NaN,False,NaN,NaN
107,25d54fa4-0afa-4e47-a1ce-e2cfe5948fda,2019-11-01,Arthrn,1,49,20,2019-10-01,1.0,True,0.0,0.0
106,25d54fa4-0afa-4e47-a1ce-e2cfe5948fda,2019-10-01,Arthrn,1,19,12,2019-09-01,1.0,True,0.0,0.0
443,9fff2f8a-21e6-47de-a2b8-7f449929d43f,2024-12-01,Drake,1,17,12,NaT,NaN,False,NaN,NaN
463,a80c5dc5-b12e-4667-9f5a-b568961f3839,2024-11-01,"Run The Jewels, El-P, Killer Mike",1,17,12,NaT,NaN,False,NaN,NaN
303,668fd73c-bf54-4310-a139-305517f05311,2026-09-01,Willie Nelson,1,13,13,NaT,NaN,False,NaN,NaN
653,f252e893-55f2-4415-9896-1aadc78d5d41,2025-12-01,Lady Parts,1,13,9,2025-11-01,1.0,True,0.0,0.0
688,ff0fd469-60c7-4bef-acd0-8f5858cc6d19,2025-12-01,Von Wegen Lisbeth,1,13,13,NaT,NaN,False,NaN,NaN
90,20244d07-534f-4eff-b4d4-930878889970,2024-11-01,Taylor Swift,1,12,5,NaT,NaN,False,NaN,NaN


In [22]:
artist_monthly_growth.to_csv(
    OUTPUT_DIR / "artist_monthly_growth.csv",
    index=False
)

print("Saved:", OUTPUT_DIR / "artist_monthly_growth.csv")

Saved: outputs/artist_monthly_growth.csv


## 8. Listener Engagement by Artist

This section measures how strongly listeners engage with each resolved artist. Artists are grouped by `artist_mbid` rather than `artist_name` to avoid separating the same artist due to spelling or naming differences.

For each artist-listener pair, the analysis calculates:

- **Returned within 30 days:** The listener played the same artist again within 30 days after their first observed listen.
- **3+ listening days:** The listener played the artist on at least three distinct dates.
- **3+ tracks played:** The listener played at least three distinct recordings by the artist.
- **Median listening days:** The median number of distinct listening dates across the artist's listeners.
- **Median tracks played:** The median number of distinct recordings across the artist's listeners.

The final artist-level dataset reports both listener counts and engagement percentages. The percentages use the artist's unique-listener count as the denominator and can be used directly in the dashboard's engagement chart.

Only listening events with a resolved MusicBrainz artist ID are included.

In [23]:
from aggregation_functions import build_artist_engagement


In [24]:
artist_engagement = build_artist_engagement(
    artist_events
)

print("Shape:", artist_engagement.shape)
artist_engagement.head(10)

Shape: (606, 11)


,artist_mbid,artist_credit,unique_listeners,returned_listeners,listeners_3plus_days,listeners_3plus_tracks,median_listening_days,median_tracks_played,return_rate_pct,share_3plus_days_pct,share_3plus_tracks_pct
0,25b7b584-d952-4662-a8b9-dd8cdfbfeb64,A$AP Rocky,2,2,0,0,1.5,2.0,100.0,0.0,0.0
1,381086ea-f511-4aba-bdf9-71c753dc5077,"Drake, Kendrick Lamar",2,2,0,0,1.5,2.0,100.0,0.0,0.0
2,561d854a-6a28-4aa7-8c99-323e6ce46c2a,Miles Davis,2,2,0,1,1.0,2.0,100.0,0.0,50.0
3,0383dadf-2a4e-4d10-a46a-e9e041da8eb3,Queen,2,1,0,0,1.5,1.5,50.0,0.0,0.0
4,272989c8-5535-492d-a25c-9f58803e027f,Kendrick Lamar,2,1,1,0,2.5,1.5,50.0,50.0,0.0
5,c8b03190-306c-4120-bb0b-6f2ebfc06ea9,"Drake, The Weeknd",2,1,0,0,1.5,1.5,50.0,0.0,0.0
6,1ee18fb3-18a6-4c7f-8ba0-bc41cdd0462e,Drake,2,0,0,0,1.0,1.0,0.0,0.0,0.0
7,300c4c73-33ac-4255-9d57-4e32627f5e13,Ringo Starr,2,0,0,0,1.0,1.0,0.0,0.0,0.0
8,38474d18-9204-4f60-a137-1fcc6750b869,A$AP Rocky,2,0,0,0,1.0,1.0,0.0,0.0,0.0
9,7bd66c18-cb33-49aa-9279-140acedbdb49,"KIDS SEE GHOSTS, Ty Dolla $ign",2,0,0,0,1.0,1.0,0.0,0.0,0.0


In [25]:
artist_engagement.to_csv(
    OUTPUT_DIR / "artist_engagement.csv",
    index=False
)

print("Saved:", OUTPUT_DIR / "artist_engagement.csv")

Saved: outputs/artist_engagement.csv
